# Coverage Model — Endpoint Test

Exercises the deployed coverage model through its external route.

The model answers one question: *given a branch this far away, at this bearing,
how much of this census tract does its isochrone cover?* It returns an overlap
proportion between 0 and 1.

Feature order matters — the server takes a bare array, not named fields:

| # | feature | notes |
|---|---------|-------|
| 0 | `distance_km` | branch to tract centroid |
| 1 | `bearing_sin` | sin of bearing in radians |
| 2 | `bearing_cos` | cos of bearing in radians |
| 3 | `tract_area_sqkm` | |
| 4 | `travel_time` | minutes |
| 5 | `mode_driving` | 1 or 0 |
| 6 | `mode_cycling` | 1 or 0 |
| 7 | `mode_walking` | 1 or 0 |

In [ ]:
import json
import math
import subprocess

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "cov-1"

FEATURES = [
    "distance_km", "bearing_sin", "bearing_cos", "tract_area_sqkm",
    "travel_time", "mode_driving", "mode_cycling", "mode_walking",
]

## Find the endpoint

The route is read from the InferenceService rather than hardcoded, so this
notebook keeps working on a different cluster or after a redeploy.

In [ ]:
def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

URL = sh(f"oc get inferenceservice {MODEL_NAME} -o jsonpath='{{.status.url}}'")
TOKEN = sh("oc whoami -t")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

print("endpoint:", URL)

## Health and auth

An unauthenticated call should be rejected — if it isn't, the endpoint is
exposed without auth and that needs fixing before anyone else uses it.

In [ ]:
anon = requests.get(f"{URL}/v2/models/{MODEL_NAME}", verify=False, timeout=30)
print(f"no token        -> {anon.status_code} {'(rejected, good)' if anon.status_code == 401 else '(AUTH NOT ENFORCED)'}")

ready = requests.get(f"{URL}/v2/health/ready", headers=HEADERS, verify=False, timeout=30)
print(f"health/ready    -> {ready.status_code}")

meta = requests.get(f"{URL}/v2/models/{MODEL_NAME}", headers=HEADERS, verify=False, timeout=30)
print(f"model metadata  -> {meta.status_code}")
print(json.dumps(meta.json(), indent=2))

## Make a prediction

In [ ]:
def predict(rows):
    """rows: list of 8-value feature lists. Returns list of predicted overlaps."""
    flat = [v for row in rows for v in row]
    body = {"inputs": [{
        "name": "predict",
        "shape": [len(rows), len(FEATURES)],
        "datatype": "FP32",
        "data": flat,
    }]}
    r = requests.post(f"{URL}/v2/models/{MODEL_NAME}/infer",
                      headers=HEADERS, json=body, verify=False, timeout=60)
    r.raise_for_status()
    return r.json()["outputs"][0]["data"]


def features(distance_km, bearing_deg=45.0, area_sqkm=1.0, travel_time=10, mode="walking"):
    rad = math.radians(bearing_deg)
    return [
        distance_km, math.sin(rad), math.cos(rad), area_sqkm, travel_time,
        1 if mode == "driving" else 0,
        1 if mode == "cycling" else 0,
        1 if mode == "walking" else 0,
    ]


print("a tract 300m from a branch, 10 min walk:")
print("  predicted overlap =", predict([features(0.3)])[0])

## How coverage decays with distance

The useful sanity check. Overlap should start high, fall off sharply, and reach
zero somewhere near the real limit for each mode — roughly 1.8 km for a 10-minute
walk, 3.2 km for a 5-minute drive.

In [ ]:
import pandas as pd

distances = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0]
modes = [("walking", 10), ("cycling", 10), ("driving", 5)]

records = []
for mode, tt in modes:
    preds = predict([features(d, travel_time=tt, mode=mode) for d in distances])
    for d, p in zip(distances, preds):
        records.append({"mode": f"{mode}/{tt}min", "distance_km": d, "predicted_overlap": p})

curve = pd.DataFrame(records).pivot(index="distance_km", columns="mode", values="predicted_overlap")
curve.round(4)

In [ ]:
ax = curve.plot(marker="o", figsize=(9, 5))
ax.set_xlabel("distance from branch (km)")
ax.set_ylabel("predicted overlap")
ax.set_title("Predicted tract coverage vs distance")
ax.axhline(0, color="#999", linewidth=0.8)
ax.grid(alpha=0.3)

## Sanity checks

These assert properties the model should hold regardless of how it was trained.
A failure here is a real finding, not a flaky test.

Note the model is a plain regressor with nothing constraining its range, so it
can emit small negative values just outside the distance where coverage stops.
The pipeline clips these afterwards; a raw caller hitting this endpoint does not.

In [ ]:
walk = predict([features(d, mode="walking", travel_time=10) for d in [0.2, 1.0, 3.0]])
near, mid, far = walk

checks = {
    "closer tract covered more than distant one": near > far,
    "monotonic decay across 0.2 / 1.0 / 3.0 km": near >= mid >= far,
    "nothing exceeds 1.0": all(p <= 1.0 for p in walk),
    "no negative predictions": all(p >= 0 for p in walk),
}

for label, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")

print()
print("predictions at 0.2 / 1.0 / 3.0 km:", [round(p, 6) for p in walk])